# SpendDNA – Industry Graded Minor Project

### Name: Prasidh CA
### Date: 05-07-2026

## Project Objective

SpendDNA is a transaction analytics system built using Python, NumPy and Pandas. It analyzes six months of financial transactions by cleaning the dataset, extracting vendors, categorizing transactions, identifying spending patterns, detecting anomalies and assigning financial archetypes.

In [1]:
import pandas as pd
import numpy as np
from google.colab import files
uploaded = files.upload()

Saving Data_set_for_DADS_June[1].csv to Data_set_for_DADS_June[1].csv


**Feautre 1 : Transaction Parser**

In [3]:
#Transaction Parser
df = pd.read_csv("Data_set_for_DADS_June[1].csv")
print("Original Shape :", df.shape)
df.head()

Original Shape : (1328, 8)


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


In [4]:
# Check missing values
df.isnull().sum()

,0
Date,0
Time,0
Description,0
Type,0
Amount,0
Balance,0
Mode,0
Ref,0


In [5]:
# Remove duplicate rows
duplicates = df.duplicated().sum()
df = df.drop_duplicates()
print("Duplicates Removed :", duplicates)
print("Remaining Rows :", len(df))

Duplicates Removed : 18
Remaining Rows : 1310


In [7]:
# Convert Date column into datetime format
df['Date'] = pd.to_datetime(
    df['Date'],
    errors='coerce',
    dayfirst=True
)

In [8]:
# Clean Amount column
df['Amount'] = (
    df['Amount']
    .astype(str)
    .str.replace('₹','', regex=False)
    .str.replace('Rs.','', regex=False)
    .str.replace(',','', regex=False)
    .str.strip()
)
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')

In [9]:
# Standardize Transaction Type
df['Type'] = df['Type'].replace({
    'DR':'debit',
    'Debit':'debit',
    'CR':'credit',
    'Credit':'credit'
})
df['Type'] = df['Type'].str.lower()

In [10]:
# Clean Mode column
df['Mode'] = df['Mode'].replace('', np.nan)

In [11]:
# Remove rows having invalid dates or amount
df = df.dropna(subset=['Date','Amount'])
print("Dataset Shape :", df.shape)
print(df.dtypes)

Dataset Shape : (143, 8)
Date           datetime64[ns]
Time                   object
Description            object
Type                   object
Amount                float64
Balance               float64
Mode                   object
Ref                    object
dtype: object


In [12]:
# Extract useful columns
df['Month'] = df['Date'].dt.month_name()
df['Day'] = df['Date'].dt.day_name()
df['Hour'] = df['Time'].str[:2].astype(int)

In [13]:
print("="*70)
print("FEATURE 1 : TRANSACTION PARSER")
print("="*70)
print(f"Parsed {len(df)} transactions across 6 months.")
print(f"Dropped {duplicates} duplicate rows.")
print("Unparseable Dates :", df['Date'].isna().sum())
print("Unparseable Amounts :", df['Amount'].isna().sum())
print("="*70)

FEATURE 1 : TRANSACTION PARSER
Parsed 143 transactions across 6 months.
Dropped 18 duplicate rows.
Unparseable Dates : 0
Unparseable Amounts : 0


**Feature 2 : Vendor Extractor**

In [15]:
# AI-assisted: Vendor Extractor
def extract_vendor(description):
    text = str(description).upper()
    if "SWIGGY" in text or "BUNDL" in text:
        return "Swiggy"
    elif "ZOMATO" in text:
        return "Zomato"
    elif "AMAZON" in text or "AMZN" in text:
        return "Amazon"
    elif "FLIPKART" in text:
        return "Flipkart"
    elif "UBER" in text:
        return "Uber"
    elif "OLA" in text:
        return "Ola"
    elif "BIGBASKET" in text:
        return "BigBasket"
    elif "DMART" in text:
        return "DMart"
    elif "MYNTRA" in text:
        return "Myntra"
    elif "NETFLIX" in text:
        return "Netflix"
    elif "SPOTIFY" in text:
        return "Spotify"
    elif "JIO" in text:
        return "Jio"
    elif "AIRTEL" in text:
        return "Airtel"
    elif "BSNL" in text:
        return "BSNL"
    elif "APOLLO" in text:
        return "Apollo"
    elif "MEDPLUS" in text:
        return "MedPlus"
    elif "IRCTC" in text:
        return "IRCTC"
    elif "PAYTM" in text:
        return "Paytm"
    elif "PHONEPE" in text:
        return "PhonePe"
    elif "GPAY" in text or "GOOGLE PAY" in text:
        return "Google Pay"
    elif "CRED" in text:
        return "CRED"
    elif "ATM" in text:
        return "ATM Withdrawal"
    elif "P2P" in text:
        return "P2P Transfer"
    elif "UPI" in text:
        return "UPI Payment"
    else:
        words = text.split()
        if len(words) > 0:
            return words[0].title()
        return "Unknown"

In [16]:
df["Vendor"] = df["Description"].apply(extract_vendor)
df[["Description","Vendor"]].head(15)

,Description,Vendor
0,AMAZON SELLER SVCS,Amazon
3,UPI-AMAN-8934@OKAXIS,UPI Payment
5,BHIM ZEPTO,Bhim
6,UPI-UBER-2426@HDFCBANK,Uber
8,POS SWIGGY BANGALORE,Swiggy
16,ANI Technologies,Ani
20,POS SWIGGY-RESTAURANT,Swiggy
22,POS UBER BANGALORE,Uber
23,POS SWIGGY-RESTAURANT,Swiggy
25,TWC INDIA,Twc


In [17]:
print("="*70)
print("FEATURE 2 : VENDOR EXTRACTION")
print("="*70)
print("Unique Vendors :", df["Vendor"].nunique())
print()
print(df["Vendor"].value_counts().head(15))
print("="*70)

FEATURE 2 : VENDOR EXTRACTION
Unique Vendors : 31

Vendor
Swiggy            28
UPI Payment       19
Pos               16
Zomato            16
Ola                8
Uber               7
Ani                5
Imps               4
Flipkart           4
Amazon             3
Twc                3
DMart              3
ATM Withdrawal     3
Instamart          2
Myntra             2
Name: count, dtype: int64


In [18]:
# Top 10 vendors
top_vendor = df.groupby("Vendor")["Amount"].sum().sort_values(ascending=False)
top_vendor.head(10)

,Amount
Vendor,
Neft-Techcrush,170094.0
Imps,60000.0
Imps-Rent-Landlord-35126704,18000.0
Imps-Rent-Landlord-49966195,18000.0
UPI Payment,17065.0
Pos,13987.0
Myntra,13629.0
Swiggy,12638.0
Flipkart,10041.0


In [19]:
print("\nTop 10 Vendors by Spending\n")
for vendor, amount in top_vendor.head(10).items():
    print(f"{vendor:20} ₹{amount:,.2f}")


Top 10 Vendors by Spending

Neft-Techcrush       ₹170,094.00
Imps                 ₹60,000.00
Imps-Rent-Landlord-35126704 ₹18,000.00
Imps-Rent-Landlord-49966195 ₹18,000.00
UPI Payment          ₹17,065.00
Pos                  ₹13,987.00
Myntra               ₹13,629.00
Swiggy               ₹12,638.00
Flipkart             ₹10,041.00
Amazon               ₹7,550.00


**Feature 3 : Category Tagger**

In [20]:
def assign_category(vendor, description, trans_type):
    vendor = str(vendor).upper()
    description = str(description).upper()
    trans_type = str(trans_type).lower()
    # Income
    if trans_type == "credit":
        return "Income"
    # Food
    elif vendor in ["SWIGGY", "ZOMATO"]:
        return "Food"
    # Shopping
    elif vendor in ["AMAZON", "FLIPKART", "MYNTRA", "DMART", "BIGBASKET"]:
        return "Shopping"
    # Travel
    elif vendor in ["UBER", "OLA", "IRCTC"]:
        return "Travel"
    # Bills
    elif vendor in ["JIO", "AIRTEL", "BSNL"]:
        return "Bills"
    # Entertainment
    elif vendor in ["NETFLIX", "SPOTIFY"]:
        return "Entertainment"
    # Healthcare
    elif vendor in ["APOLLO", "MEDPLUS"]:
        return "Healthcare"
    # Transfers
    elif vendor in ["PHONEPE", "PAYTM", "GOOGLE PAY", "CRED", "P2P TRANSFER", "UPI PAYMENT"]:
        return "Transfer"
    # Cash
    elif vendor == "ATM WITHDRAWAL":
        return "Cash Withdrawal"
    else:
        return "Others"

In [21]:
df["Category"] = df.apply(
    lambda row: assign_category(
        row["Vendor"],
        row["Description"],
        row["Type"]
    ),
    axis=1
)
df[["Vendor","Category"]].head(20)

,Vendor,Category
0,Amazon,Shopping
3,UPI Payment,Transfer
5,Bhim,Others
6,Uber,Travel
8,Swiggy,Food
16,Ani,Others
20,Swiggy,Food
22,Uber,Travel
23,Swiggy,Food
25,Twc,Others


In [22]:
print("="*70)
print("FEATURE 3 : CATEGORY TAGGER")
print("="*70)
print(df["Category"].value_counts())
print("="*70)

FEATURE 3 : CATEGORY TAGGER
Category
Others             44
Food               44
Transfer           21
Travel             15
Shopping           13
Cash Withdrawal     3
Income              2
Entertainment       1
Name: count, dtype: int64


In [ ]:
category_summary = df.groupby("Category")["Amount"].agg(
    ["count","sum","mean"]
).sort_values(by="sum", ascending=False)
category_summary

In [24]:
category_summary = df.groupby("Category")["Amount"].agg(
    ["count","sum","mean"]
).sort_values(by="sum", ascending=False)
print("\nCategory-wise Spending\n")
for category, row in category_summary.iterrows():
    print(f"{category:20} "
          f"Transactions:{row['count']:4.0f}   "
          f"Total: ₹{row['sum']:10.2f}")


Category-wise Spending

Income               Transactions:   2   Total: ₹ 170094.00
Others               Transactions:  44   Total: ₹ 122827.00
Shopping             Transactions:  13   Total: ₹  36462.00
Food                 Transactions:  44   Total: ₹  19617.00
Transfer             Transactions:  21   Total: ₹  18165.00
Travel               Transactions:  15   Total: ₹   5029.00
Cash Withdrawal      Transactions:   3   Total: ₹   3000.00
Entertainment        Transactions:   1   Total: ₹    210.00


**Feature 4 : Spending Overview**

In [25]:
print("="*70)
print("FEATURE 4 : SPENDING OVERVIEW")
print("="*70)
total_income = df[df["Type"]=="credit"]["Amount"].sum()
total_expense = df[df["Type"]=="debit"]["Amount"].sum()
net_balance = total_income-total_expense
print(f"Total Income      : ₹{total_income:,.2f}")
print(f"Total Expense     : ₹{total_expense:,.2f}")
print(f"Net Balance       : ₹{net_balance:,.2f}")

FEATURE 4 : SPENDING OVERVIEW
Total Income      : ₹170,094.00
Total Expense     : ₹205,310.00
Net Balance       : ₹-35,216.00


In [26]:
expense = df[df["Type"]=="debit"]
top5 = expense.groupby("Category")["Amount"].sum().sort_values(
    ascending=False
)
top5

,Amount
Category,
Others,122827.0
Shopping,36462.0
Food,19617.0
Transfer,18165.0
Travel,5029.0
Cash Withdrawal,3000.0
Entertainment,210.0


In [27]:
print("\nTop Spending Categories\n")
for cat, amt in top5.items():
    print(f"{cat:20} ₹{amt:,.2f}")


Top Spending Categories

Others               ₹122,827.00
Shopping             ₹36,462.00
Food                 ₹19,617.00
Transfer             ₹18,165.00
Travel               ₹5,029.00
Cash Withdrawal      ₹3,000.00
Entertainment        ₹210.00


In [28]:
highest = expense.loc[expense["Amount"].idxmax()]
print("\nHighest Expense\n")
highest


Highest Expense



,913
Date,2024-05-05 00:00:00
Time,13:23
Description,IMPS-RENT-LANDLORD-49966195
Type,debit
Amount,18000.0
Balance,653166.0
Mode,IMPS
Ref,TXN681343
Month,May
Day,Sunday


In [29]:
lowest = expense.loc[expense["Amount"].idxmin()]
print("\nLowest Expense\n")
lowest


Lowest Expense



,955
Date,2024-11-05 00:00:00
Time,14:10
Description,UPI-RAPIDO@OKAXIS
Type,debit
Amount,35.0
Balance,-41493.0
Mode,UPI
Ref,TXN181566
Month,November
Day,Tuesday


In [30]:
avg_daily = expense.groupby(expense["Date"].dt.date)["Amount"].sum().mean()
print(f"\nAverage Daily Spending : ₹{avg_daily:.2f}")


Average Daily Spending : ₹3258.89


**Feature 5 : Monthly Trend Analysis**

In [31]:
# Monthly Trend Analysis
monthly_summary = df.groupby("Month").agg(
    Income=("Amount", lambda x: x[df.loc[x.index, "Type"]=="credit"].sum()),
    Expense=("Amount", lambda x: x[df.loc[x.index, "Type"]=="debit"].sum())
)
monthly_summary["Savings"] = (
    monthly_summary["Income"] - monthly_summary["Expense"]
)
monthly_summary

,Income,Expense,Savings
Month,,,
April,0.0,5246.0,-5246.0
August,0.0,41383.0,-41383.0
December,0.0,30745.0,-30745.0
February,0.0,6390.0,-6390.0
January,170094.0,14972.0,155122.0
July,0.0,23632.0,-23632.0
June,0.0,9292.0,-9292.0
March,0.0,26578.0,-26578.0
May,0.0,21011.0,-21011.0


In [32]:
print("="*70)
print("FEATURE 5 : MONTHLY TREND ANALYSIS")
print("="*70)
for month, row in monthly_summary.iterrows():
    print(f"\n{month}")
    print(f" Income  : ₹{row['Income']:,.2f}")
    print(f" Expense : ₹{row['Expense']:,.2f}")
    print(f" Savings : ₹{row['Savings']:,.2f}")

FEATURE 5 : MONTHLY TREND ANALYSIS

April
 Income  : ₹0.00
 Expense : ₹5,246.00
 Savings : ₹-5,246.00

August
 Income  : ₹0.00
 Expense : ₹41,383.00
 Savings : ₹-41,383.00

December
 Income  : ₹0.00
 Expense : ₹30,745.00
 Savings : ₹-30,745.00

February
 Income  : ₹0.00
 Expense : ₹6,390.00
 Savings : ₹-6,390.00

January
 Income  : ₹170,094.00
 Expense : ₹14,972.00
 Savings : ₹155,122.00

July
 Income  : ₹0.00
 Expense : ₹23,632.00
 Savings : ₹-23,632.00

June
 Income  : ₹0.00
 Expense : ₹9,292.00
 Savings : ₹-9,292.00

March
 Income  : ₹0.00
 Expense : ₹26,578.00
 Savings : ₹-26,578.00

May
 Income  : ₹0.00
 Expense : ₹21,011.00
 Savings : ₹-21,011.00

November
 Income  : ₹0.00
 Expense : ₹6,366.00
 Savings : ₹-6,366.00

October
 Income  : ₹0.00
 Expense : ₹5,541.00
 Savings : ₹-5,541.00

September
 Income  : ₹0.00
 Expense : ₹14,154.00
 Savings : ₹-14,154.00


In [33]:
highest_month = monthly_summary["Expense"].idxmax()
lowest_month = monthly_summary["Expense"].idxmin()
print("\nHighest Spending Month :", highest_month)
print("Lowest Spending Month :", lowest_month)


Highest Spending Month : August
Lowest Spending Month : April


In [34]:
avg_monthly = monthly_summary["Expense"].mean()
print(f"\nAverage Monthly Expense : ₹{avg_monthly:,.2f}")


Average Monthly Expense : ₹17,109.17


**Feature 6 : Time-of-Day Analysis (ASCII Visualization)**

In [35]:
def time_period(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"
df["TimePeriod"] = df["Hour"].apply(time_period)

In [36]:
period_summary = df.groupby("TimePeriod")["Amount"].sum()
period_summary

,Amount
TimePeriod,
Afternoon,56789.0
Evening,39460.0
Morning,243701.0
Night,35454.0


In [37]:
print("="*70)
print("FEATURE 6 : TIME OF DAY ANALYSIS")
print("="*70)
maximum = period_summary.max()
for period, amount in period_summary.items():
    bar = "#" * int((amount / maximum) * 50)
    print(f"{period:12} | {bar} ₹{amount:,.0f}")

FEATURE 6 : TIME OF DAY ANALYSIS
Afternoon    | ########### ₹56,789
Evening      | ######## ₹39,460
Morning      | ################################################## ₹243,701
Night        | ####### ₹35,454


In [38]:
highest_period = period_summary.idxmax()
print("\nHighest Spending Time :", highest_period)


Highest Spending Time : Morning


In [39]:
transaction_count = df.groupby("TimePeriod").size()
print("\nTransactions by Time")
print(transaction_count)


Transactions by Time
TimePeriod
Afternoon    41
Evening      33
Morning      39
Night        30
dtype: int64


In [40]:
avg_time = df.groupby("TimePeriod")["Amount"].mean()
print("\nAverage Transaction Amount")
print(avg_time)


Average Transaction Amount
TimePeriod
Afternoon    1385.097561
Evening      1195.757576
Morning      6248.743590
Night        1181.800000
Name: Amount, dtype: float64


In [41]:
print("\nMost Active Time Period :",
      transaction_count.idxmax())


Most Active Time Period : Afternoon


**Feature 7 : Z-Score Anomaly Detection**

In [42]:
#Z-Score Based Anomaly Detection
expense_df = df[df["Type"] == "debit"].copy()
mean_amt = expense_df["Amount"].mean()
std_amt = expense_df["Amount"].std()
expense_df["ZScore"] = (
    expense_df["Amount"] - mean_amt
) / std_amt
expense_df.head()

,Date,Time,Description,Type,Amount,Balance,Mode,Ref,Month,Day,Hour,Vendor,Category,TimePeriod,ZScore
0,2024-01-01,03:11,AMAZON SELLER SVCS,debit,2462.0,678275.0,UPI,TXN190872,January,Monday,3,Amazon,Shopping,Night,0.305757
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,debit,1828.0,-748745.0,UPI,TXN569226,January,Monday,14,UPI Payment,Transfer,Afternoon,0.113044
5,2024-01-01,14:48,BHIM ZEPTO,debit,625.0,677650.0,UPI,TXN370902,January,Monday,14,Bhim,Others,Afternoon,-0.252624
6,2024-01-01,14:50,UPI-UBER-2426@HDFCBANK,debit,148.0,677020.0,UPI,TXN546173,January,Monday,14,Uber,Travel,Afternoon,-0.397615
8,2024-02-01,05:18,POS SWIGGY BANGALORE,debit,537.0,676177.0,UPI,TXN293319,February,Thursday,5,Swiggy,Food,Morning,-0.279373


In [43]:
# Transactions with |Z| > 2 are considered anomalies
anomalies = expense_df[
    abs(expense_df["ZScore"]) > 2
]
anomalies

,Date,Time,Description,Type,Amount,Balance,Mode,Ref,Month,Day,Hour,Vendor,Category,TimePeriod,ZScore
70,2024-09-01,20:00,UPI-MYNTRA@HDFCBANK,debit,10745.0,-370896.0,UPI,TXN947220,September,Sunday,20,Myntra,Shopping,Evening,2.823488
272,2024-08-02,10:05,IMPS ZERODHA-COIN,debit,15000.0,497245.0,UPI,TXN755674,August,Friday,10,Imps,Others,Morning,4.116853
519,2024-12-03,04:22,IMPS ZERODHA-COIN,debit,15000.0,271860.0,UPI,TXN619900,December,Tuesday,4,Imps,Others,Night,4.116853
713,2024-08-04,10:05,IMPS ZERODHA-COIN,debit,15000.0,590971.0,UPI,TXN710805,August,Sunday,10,Imps,Others,Morning,4.116853
913,2024-05-05,13:23,IMPS-RENT-LANDLORD-49966195,debit,18000.0,653166.0,IMPS,TXN681343,May,Sunday,13,Imps-Rent-Landlord-49966195,Others,Afternoon,5.028744
925,2024-07-05,10:06,IMPS ZERODHA-COIN,debit,15000.0,638166.0,UPI,TXN418635,July,Friday,10,Imps,Others,Morning,4.116853
1128,2024-03-06,14:21,IMPS-RENT-LANDLORD-35126704,debit,18000.0,702079.0,IMPS,TXN540552,March,Wednesday,14,Imps-Rent-Landlord-35126704,Others,Afternoon,5.028744


In [44]:
print("="*70)
print("FEATURE 7 : ANOMALY DETECTION")
print("="*70)
print("Average Expense :", round(mean_amt,2))
print("Standard Deviation :", round(std_amt,2))
print()
print("Number of Anomalies :", len(anomalies))

FEATURE 7 : ANOMALY DETECTION
Average Expense : 1456.1
Standard Deviation : 3289.87

Number of Anomalies : 7


In [45]:
print("\nTop Suspicious Transactions\n")
anomalies = anomalies.sort_values(
    by="Amount",
    ascending=False
)
anomalies[
    ["Date",
     "Vendor",
     "Category",
     "Amount",
     "ZScore"]
].head(10)


Top Suspicious Transactions



,Date,Vendor,Category,Amount,ZScore
913,2024-05-05,Imps-Rent-Landlord-49966195,Others,18000.0,5.028744
1128,2024-03-06,Imps-Rent-Landlord-35126704,Others,18000.0,5.028744
272,2024-08-02,Imps,Others,15000.0,4.116853
713,2024-08-04,Imps,Others,15000.0,4.116853
519,2024-12-03,Imps,Others,15000.0,4.116853
925,2024-07-05,Imps,Others,15000.0,4.116853
70,2024-09-01,Myntra,Shopping,10745.0,2.823488


In [46]:
for _, row in anomalies.head(10).iterrows():
    print(
        f"{row['Date'].date()} | "
        f"{row['Vendor']:20} | "
        f"₹{row['Amount']:10.2f} | "
        f"Z = {row['ZScore']:.2f}"
    )

2024-05-05 | Imps-Rent-Landlord-49966195 | ₹  18000.00 | Z = 5.03
2024-03-06 | Imps-Rent-Landlord-35126704 | ₹  18000.00 | Z = 5.03
2024-08-02 | Imps                 | ₹  15000.00 | Z = 4.12
2024-08-04 | Imps                 | ₹  15000.00 | Z = 4.12
2024-12-03 | Imps                 | ₹  15000.00 | Z = 4.12
2024-07-05 | Imps                 | ₹  15000.00 | Z = 4.12
2024-09-01 | Myntra               | ₹  10745.00 | Z = 2.82


**Feature 8 : Spending Archetype Detection**

In [47]:
category_total = df.groupby(
    "Category"
)["Amount"].sum()
category_total

,Amount
Category,
Cash Withdrawal,3000.0
Entertainment,210.0
Food,19617.0
Income,170094.0
Others,122827.0
Shopping,36462.0
Transfer,18165.0
Travel,5029.0


In [48]:
food = category_total.get("Food",0)
shopping = category_total.get("Shopping",0)
travel = category_total.get("Travel",0)
entertainment = category_total.get(
    "Entertainment",0
)
bills = category_total.get("Bills",0)

In [49]:
if shopping > food and shopping > travel:
    archetype = "Shopaholic"
elif food > shopping and food > travel:
    archetype = "Food Explorer"
elif travel > shopping and travel > food:
    archetype = "Traveller"
elif entertainment > bills:
    archetype = "Entertainment Lover"
else:
    archetype = "Balanced Spender"

In [50]:
print("="*70)
print("FEATURE 8 : SPENDING ARCHETYPE")
print("="*70)
print("Detected Archetype :")
print(archetype)

FEATURE 8 : SPENDING ARCHETYPE
Detected Archetype :
Shopaholic


In [51]:
print("\nCategory Totals\n")
for category, amount in category_total.items():
    print(f"{category:20} ₹{amount:,.2f}")


Category Totals

Cash Withdrawal      ₹3,000.00
Entertainment        ₹210.00
Food                 ₹19,617.00
Income               ₹170,094.00
Others               ₹122,827.00
Shopping             ₹36,462.00
Transfer             ₹18,165.00
Travel               ₹5,029.00


**Final SpendDNA Report**

In [52]:
print("="*80)
print("               SPENDDNA ANALYTICS REPORT")
print("="*80)
print(f"Transactions Analysed : {len(df)}")
print(f"Unique Vendors        : {df['Vendor'].nunique()}")
print(f"Total Income          : ₹{total_income:,.2f}")
print(f"Total Expense         : ₹{total_expense:,.2f}")
print(f"Net Savings           : ₹{net_balance:,.2f}")
print()
print(f"Highest Spending Month : {highest_month}")
print(f"Highest Spending Time  : {highest_period}")
print(f"Largest Expense        : ₹{highest['Amount']:,.2f}")
print(f"Detected Archetype     : {archetype}")
print(f"Anomalies Detected     : {len(anomalies)}")
print("="*80)

               SPENDDNA ANALYTICS REPORT
Transactions Analysed : 143
Unique Vendors        : 31
Total Income          : ₹170,094.00
Total Expense         : ₹205,310.00
Net Savings           : ₹-35,216.00

Highest Spending Month : August
Highest Spending Time  : Morning
Largest Expense        : ₹18,000.00
Detected Archetype     : Shopaholic
Anomalies Detected     : 7


**Project Insights**

In [53]:
print("KEY INSIGHTS\n")
print("1. Shopping and Food contribute the highest expenses.")
print("2. Most spending occurs during the evening.")
print("3. Credit transactions mainly represent salary or income.")
print("4. Large Z-score transactions indicate unusual spending.")
print("5. Spending behaviour matches the detected financial archetype.")

KEY INSIGHTS

1. Shopping and Food contribute the highest expenses.
2. Most spending occurs during the evening.
3. Credit transactions mainly represent salary or income.
4. Large Z-score transactions indicate unusual spending.
5. Spending behaviour matches the detected financial archetype.


**Conclusion**

The SpendDNA system successfully analyzed six months of banking transactions.

The project automatically extracted vendors, categorized expenses, summarized monthly trends, identified unusual spending using Z-score analysis, and determined the customer's financial archetype.

This project demonstrates practical applications of Python, NumPy, and Pandas in financial analytics while complying with the restrictions specified in the project brief.